In [3]:
import sqlite3
import random
import json
import os

# Path to your MIMIC-III SQLite database
DB_PATH = "../Assignment_MIMIC_SQL/database.db"
OUTPUT_DIR = "./generated_prompts"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Connect to the database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Fetch a random clinical note
def get_random_note():
    cursor.execute("SELECT TEXT FROM NOTEEVENTS WHERE LENGTH(TEXT) > 100 LIMIT 1000")
    notes = cursor.fetchall()
    return random.choice(notes)[0] if notes else None

# Prompt templates
def zero_shot_prompt(note):
    return f"""
You are a medical assistant. Simplify the following clinical note into plain English so that a patient with no medical training can easily understand it.

Clinical Note:
\"\"\"{note}\"\"\"

Simplified Version:
"""

def few_shot_prompt(note):
    return f"""
You are a medical assistant. Simplify clinical notes into plain English for patients.

Example 1:
Clinical Note: "The patient was diagnosed with pneumonia based on chest X-ray showing lung infiltrates."
Simplified Version: "You have a lung infection called pneumonia. It was found on your chest X-ray."

Example 2:
Clinical Note: "The patient is experiencing hyperglycemia secondary to poorly controlled diabetes mellitus type II."
Simplified Version: "Your blood sugar is too high because your diabetes hasn't been well controlled."

Now simplify this note:

Clinical Note:
\"\"\"{note}\"\"\"

Simplified Version:
"""

def chain_of_thought_prompt(note):
    return f"""
You are a medical assistant. Simplify clinical notes into plain English for patients.  
First, list the key medical terms and their meanings. Then rewrite the note.

Clinical Note:
\"\"\"{note}\"\"\"

Step 1: Identify terms and meanings.
- Term 1: Meaning
- Term 2: Meaning
- ...

Step 2: Simplified Version:
"""

def reasoning_and_diagnosis_prompt(note):
    return f"""
You are a medical assistant AI. Given a clinical note, identify the most likely diagnosis and explain your reasoning.  

Clinical Note:
\"\"\"{note}\"\"\"

Your Reasoning:
1. Analyze the symptoms and context.
2. Match symptoms to a likely diagnosis.
3. Output the diagnosis and rationale.

Most likely diagnosis:
"""

# Generate and export prompts to a JSON file
def export_prompts(note_id, note_text):
    prompts = {
        "note_id": note_id,
        "original_note": note_text,
        "zero_shot": zero_shot_prompt(note_text),
        "few_shot": few_shot_prompt(note_text),
        "chain_of_thought": chain_of_thought_prompt(note_text),
        "reasoning_and_diagnosis": reasoning_and_diagnosis_prompt(note_text)
    }
    
    filename = os.path.join(OUTPUT_DIR, f"note_{note_id}.json")
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(prompts, f, indent=4)
    print(f"✅ Prompts saved to: {filename}")

# Pull a note and generate prompts
cursor.execute("SELECT ROWID, TEXT FROM NOTEEVENTS WHERE LENGTH(TEXT) > 100 LIMIT 1000")
notes = cursor.fetchall()

if notes:
    random_note = random.choice(notes)
    note_id, note_text = random_note[0], random_note[1]
    export_prompts(note_id, note_text)
else:
    print("No clinical notes found in the database!")

# Close connection
conn.close()


✅ Prompts saved to: ./generated_prompts/note_856.json


In [ ]:
# After generating the prompt dictionary...
def export_prompts(note_id, note_text):
    prompts = {
        "note_id": note_id,
        "original_note": note_text,
        "zero_shot": zero_shot_prompt(note_text),
        "few_shot": few_shot_prompt(note_text),
        "chain_of_thought": chain_of_thought_prompt(note_text),
        "reasoning_and_diagnosis": reasoning_and_diagnosis_prompt(note_text)
    }
    
    # 💡 Print a quick preview in console:
    print("\n===================================")
    print(f"📝 Note ID: {note_id}")
    print("🔹 Zero-Shot Prompt:\n", prompts["zero_shot"][:500] + "...\n")
    print("🔹 Few-Shot Prompt:\n", prompts["few_shot"][:500] + "...\n")
    print("🔹 Chain-of-Thought Prompt:\n", prompts["chain_of_thought"][:500] + "...\n")
    print("🔹 Reasoning & Diagnosis Prompt:\n", prompts["reasoning_and_diagnosis"][:500] + "...\n")
    print("===================================\n")

    # Save to JSON
    filename = os.path.join(OUTPUT_DIR, f"note_{note_id}.json")
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(prompts, f, indent=4)
    print(f"✅ Prompts saved to: {filename}")
